# Predict Customer Churn
## Score: .91437

In [47]:
N_FOLDS = 10
N_SEEDS = 5
OPTUNA_TRIALS = 80
USE_RF_ET = False

In [48]:
import numpy as np
import pandas as pd

train = pd.read_csv("playground-series-s6e3/train.csv")
test = pd.read_csv("playground-series-s6e3/test.csv")
original = pd.read_csv("original_data/WA_Fn-UseC_-Telco-Customer-Churn.csv")
original = original.drop(columns=['customerID'])
original = original[train.columns.drop('id')]
train_combined = pd.concat([train.drop(columns=['id']), original], ignore_index=True)
sample_weights = np.array([1.0] * len(train) + [0.5] * len(original))
y_full = train_combined['Churn'].map({'Yes': 1, 'No': 0})
X_full = train_combined.drop(columns=['Churn'])
test_ids = test['id']
X_test_raw = test.drop(columns=['id'])
print(f"Train: {len(train)}, Original: {len(original)}, Combined: {len(X_full)}")

Train: 594194, Original: 7043, Combined: 601237


In [49]:
def engineer_features(df, total_charges_median=None):
    df = df.copy()
    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
    median_tc = total_charges_median if total_charges_median is not None else df['TotalCharges'].median()
    df['TotalCharges'] = df['TotalCharges'].fillna(median_tc)
    df['AvgChargesPerMonth'] = df['TotalCharges'] / df['tenure'].replace(0, 1)
    df['MonthlyChargesSqrt'] = np.sqrt(df['MonthlyCharges'])
    df['tenure_squared'] = df['tenure'] ** 2
    df['ChargesRatio'] = df['TotalCharges'] / (df['MonthlyCharges'] * (df['tenure'] + 1))
    df['Contract_MonthToMonth'] = (df['Contract'] == 'Month-to-month').astype(int)
    df['Contract_OneYear'] = (df['Contract'] == 'One year').astype(int)
    df['Contract_TwoYear'] = (df['Contract'] == 'Two year').astype(int)
    df['IsFiberOptic'] = (df['InternetService'] == 'Fiber optic').astype(int)
    df['NumStreaming'] = ((df['StreamingTV'] == 'Yes') | (df['StreamingMovies'] == 'Yes')).astype(int)
    df['NumStreamingBoth'] = ((df['StreamingTV'] == 'Yes') & (df['StreamingMovies'] == 'Yes')).astype(int)
    addon_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport']
    df['NumAddons'] = sum((df[c] == 'Yes').astype(int) for c in addon_cols)
    df['PaymentElectronic'] = (df['PaymentMethod'] == 'Electronic check').astype(int)
    df['HasDependents'] = (df['Dependents'] == 'Yes').astype(int)
    df['HasPartner'] = (df['Partner'] == 'Yes').astype(int)
    df['SeniorWithShortTenure'] = (df['SeniorCitizen'] == 1) & (df['tenure'] < 12)
    df['SeniorWithShortTenure'] = df['SeniorWithShortTenure'].astype(int)
    df['HighChargeShortTenure'] = (df['MonthlyCharges'] > 70) & (df['tenure'] < 12)
    df['HighChargeShortTenure'] = df['HighChargeShortTenure'].astype(int)
    return df

def target_encode(train_df, test_df, col, target_series, m=5):
    global_mean = target_series.mean()
    combined = pd.DataFrame({col: train_df[col], '_y': target_series.values})
    agg = combined.groupby(col)['_y'].agg(['mean', 'count'])
    smooth = (agg['mean'] * agg['count'] + global_mean * m) / (agg['count'] + m)
    train_df = train_df.copy()
    test_df = test_df.copy()
    train_df[f'{col}_te'] = train_df[col].map(smooth).fillna(global_mean)
    test_df[f'{col}_te'] = test_df[col].map(smooth).fillna(global_mean)
    return train_df, test_df

X_full = engineer_features(X_full)
tc_median = X_full['TotalCharges'].median()
X_test_raw = engineer_features(X_test_raw, total_charges_median=tc_median)

for col in ['Contract', 'PaymentMethod', 'InternetService']:
    X_full, X_test_raw = target_encode(X_full, X_test_raw, col, y_full, m=20)
    X_full = X_full.drop(columns=[col])
    X_test_raw = X_test_raw.drop(columns=[col])

X_encoded = pd.get_dummies(X_full, drop_first=True)
X_test_encoded = pd.get_dummies(X_test_raw, drop_first=True)
X_test_encoded = X_test_encoded.reindex(columns=X_encoded.columns, fill_value=0)

In [50]:
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
fit_kw = {'sample_weight': sample_weights}

In [51]:
import optuna
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score

def xgb_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 300, 800),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.15),
        'subsample': trial.suggest_float('subsample', 0.6, 0.95),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 0.95),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1.0, 4.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
    }
    scores = []
    for train_idx, val_idx in cv.split(X_encoded, y_full):
        X_tr, X_val = X_encoded.iloc[train_idx], X_encoded.iloc[val_idx]
        y_tr, y_val = y_full.iloc[train_idx], y_full.iloc[val_idx]
        sw_tr = sample_weights[train_idx]
        m = XGBClassifier(**params, random_state=42, eval_metric='logloss')
        m.fit(X_tr, y_tr, sample_weight=sw_tr)
        scores.append(roc_auc_score(y_val, m.predict_proba(X_val)[:, 1]))
    return np.mean(scores)

study_xgb = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(n_startup_trials=10))
study_xgb.optimize(xgb_objective, n_trials=OPTUNA_TRIALS, show_progress_bar=True)
best_xgb_params = study_xgb.best_params
print(f"XGB Best CV AUC: {study_xgb.best_value:.4f}")

[I 2026-03-05 01:21:05,603] A new study created in memory with name: no-name-3030c17d-d07c-4e86-a110-a50f7c7fc050


  0%|          | 0/80 [00:00<?, ?it/s]

[I 2026-03-05 01:22:22,407] Trial 0 finished with value: 0.9155817459050464 and parameters: {'n_estimators': 473, 'max_depth': 4, 'learning_rate': 0.11480184542978879, 'subsample': 0.8302551073126208, 'colsample_bytree': 0.9371755802679611, 'scale_pos_weight': 3.890638592405761, 'reg_alpha': 0.009503288832071678, 'reg_lambda': 0.021586686849788803, 'min_child_weight': 4}. Best is trial 0 with value: 0.9155817459050464.
[I 2026-03-05 01:24:54,760] Trial 1 finished with value: 0.9156192667033725 and parameters: {'n_estimators': 757, 'max_depth': 4, 'learning_rate': 0.13248070860387953, 'subsample': 0.7136131961175431, 'colsample_bytree': 0.6058913661244648, 'scale_pos_weight': 3.9273355942273898, 'reg_alpha': 0.0047137653059138454, 'reg_lambda': 1.0844903763477423, 'min_child_weight': 2}. Best is trial 1 with value: 0.9156192667033725.
[I 2026-03-05 01:26:45,729] Trial 2 finished with value: 0.9155313861112836 and parameters: {'n_estimators': 381, 'max_depth': 7, 'learning_rate': 0.07323

In [52]:
from lightgbm import LGBMClassifier

def lgb_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 300, 800),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.15),
        'subsample': trial.suggest_float('subsample', 0.6, 0.95),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 0.95),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1.0, 4.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
    }
    scores = []
    for train_idx, val_idx in cv.split(X_encoded, y_full):
        X_tr, X_val = X_encoded.iloc[train_idx], X_encoded.iloc[val_idx]
        y_tr, y_val = y_full.iloc[train_idx], y_full.iloc[val_idx]
        sw_tr = sample_weights[train_idx]
        m = LGBMClassifier(**params, random_state=42, verbose=-1)
        m.fit(X_tr, y_tr, sample_weight=sw_tr)
        scores.append(roc_auc_score(y_val, m.predict_proba(X_val)[:, 1]))
    return np.mean(scores)

study_lgb = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(n_startup_trials=10))
study_lgb.optimize(lgb_objective, n_trials=OPTUNA_TRIALS, show_progress_bar=True)
best_lgb_params = study_lgb.best_params
print(f"LGB Best CV AUC: {study_lgb.best_value:.4f}")

[I 2026-03-05 04:15:51,295] A new study created in memory with name: no-name-bb8c2823-542e-4dc5-9a76-75a68ec05d85


  0%|          | 0/80 [00:00<?, ?it/s]

[I 2026-03-05 04:17:59,988] Trial 0 finished with value: 0.9150544264759232 and parameters: {'n_estimators': 799, 'max_depth': 7, 'learning_rate': 0.09606753290517765, 'subsample': 0.8128003541487884, 'colsample_bytree': 0.9141874365185068, 'scale_pos_weight': 1.131701847301108, 'reg_alpha': 0.03971472595946077, 'reg_lambda': 0.002412272055635826, 'min_child_samples': 5}. Best is trial 0 with value: 0.9150544264759232.
[I 2026-03-05 04:19:31,178] Trial 1 finished with value: 0.9155245129321392 and parameters: {'n_estimators': 510, 'max_depth': 8, 'learning_rate': 0.08311449315814103, 'subsample': 0.6873381437698594, 'colsample_bytree': 0.7040109580446905, 'scale_pos_weight': 3.592352812170101, 'reg_alpha': 0.6842858556965414, 'reg_lambda': 0.24886342373502085, 'min_child_samples': 58}. Best is trial 1 with value: 0.9155245129321392.
[I 2026-03-05 04:21:30,429] Trial 2 finished with value: 0.9153697658990223 and parameters: {'n_estimators': 761, 'max_depth': 4, 'learning_rate': 0.043778

In [53]:
from catboost import CatBoostClassifier

def cat_objective(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 300, 800),
        'depth': trial.suggest_int('depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.15),
        'subsample': trial.suggest_float('subsample', 0.6, 0.95),
        'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.6, 0.95),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1.0, 4.0),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-3, 10.0, log=True),
    }
    scores = []
    for train_idx, val_idx in cv.split(X_encoded, y_full):
        X_tr, X_val = X_encoded.iloc[train_idx], X_encoded.iloc[val_idx]
        y_tr, y_val = y_full.iloc[train_idx], y_full.iloc[val_idx]
        sw_tr = sample_weights[train_idx]
        m = CatBoostClassifier(**params, random_state=42, verbose=0)
        m.fit(X_tr, y_tr, sample_weight=sw_tr)
        scores.append(roc_auc_score(y_val, m.predict_proba(X_val)[:, 1]))
    return np.mean(scores)

study_cat = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(n_startup_trials=10))
study_cat.optimize(cat_objective, n_trials=OPTUNA_TRIALS, show_progress_bar=True)
best_cat_params = study_cat.best_params
print(f"CatBoost Best CV AUC: {study_cat.best_value:.4f}")

[I 2026-03-05 07:04:08,062] A new study created in memory with name: no-name-4c839e8d-a0df-4ae1-9f7d-d192203156f7


  0%|          | 0/80 [00:00<?, ?it/s]

[I 2026-03-05 07:08:21,180] Trial 0 finished with value: 0.9154955954935413 and parameters: {'iterations': 613, 'depth': 5, 'learning_rate': 0.10091397093621658, 'subsample': 0.7902757652104319, 'colsample_bylevel': 0.7542225703942237, 'scale_pos_weight': 3.5636657219851617, 'l2_leaf_reg': 0.01856381156014518}. Best is trial 0 with value: 0.9154955954935413.
[I 2026-03-05 07:10:22,967] Trial 1 finished with value: 0.9152357600943601 and parameters: {'iterations': 301, 'depth': 5, 'learning_rate': 0.1448061001797119, 'subsample': 0.7150155555117699, 'colsample_bylevel': 0.7298676852191772, 'scale_pos_weight': 2.9108156205374076, 'l2_leaf_reg': 0.0031270373282731244}. Best is trial 0 with value: 0.9154955954935413.
[I 2026-03-05 07:14:36,972] Trial 2 finished with value: 0.9152748648837298 and parameters: {'iterations': 640, 'depth': 5, 'learning_rate': 0.05618693398994287, 'subsample': 0.8979620943572255, 'colsample_bylevel': 0.7552074162547007, 'scale_pos_weight': 2.003370737440529, 'l

In [54]:
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import roc_auc_score
from scipy.optimize import minimize

n_models = 3 if not USE_RF_ET else 5
oof = np.zeros((len(X_encoded), n_models))
test_preds = np.zeros((len(X_test_encoded), n_models))

def get_models(seed, fold):
    models = [
        XGBClassifier(**best_xgb_params, random_state=seed+fold, eval_metric='logloss'),
        LGBMClassifier(**best_lgb_params, random_state=seed+fold, verbose=-1),
        CatBoostClassifier(**best_cat_params, random_state=seed+fold, verbose=0),
    ]
    if USE_RF_ET:
        models.extend([
            RandomForestClassifier(n_estimators=300, max_depth=12, class_weight='balanced', random_state=seed+fold),
            ExtraTreesClassifier(n_estimators=300, max_depth=12, class_weight='balanced', random_state=seed+fold),
        ])
    return models

for fold, (train_idx, val_idx) in enumerate(cv.split(X_encoded, y_full)):
    X_tr, X_val = X_encoded.iloc[train_idx], X_encoded.iloc[val_idx]
    y_tr = y_full.iloc[train_idx]
    sw_tr = sample_weights[train_idx]
    models = get_models(42, fold)
    for i, m in enumerate(models):
        m.fit(X_tr, y_tr, sample_weight=sw_tr)
        oof[val_idx, i] = m.predict_proba(X_val)[:, 1]
        test_preds[:, i] += m.predict_proba(X_test_encoded)[:, 1] / N_FOLDS

def neg_auc(w):
    blend = oof @ w
    return -roc_auc_score(y_full, blend)

best_auc = -1
best_w = None
for x0 in [np.ones(n_models)/n_models, np.array([0.5, 0.3, 0.2] if n_models==3 else [0.2]*5)]:
    result = minimize(neg_auc, x0=x0, method='SLSQP',
                      bounds=[(0, 1)]*n_models,
                      constraints={'type': 'eq', 'fun': lambda w: np.sum(w) - 1})
    auc = -result.fun
    if auc > best_auc:
        best_auc = auc
        best_w = result.x
blend_weights = best_w
blend_oof = oof @ blend_weights
names = ['XGB','LGB','Cat'] + (['RF','ET'] if USE_RF_ET else [])
print(f"Blend OOF AUC: {roc_auc_score(y_full, blend_oof):.4f}")
print(f"Weights: {dict(zip(names, np.round(blend_weights, 4)))}")

Blend OOF AUC: 0.9162
Weights: {'XGB': np.float64(0.3333), 'LGB': np.float64(0.3333), 'Cat': np.float64(0.3333)}


In [55]:
all_test_preds = [test_preds @ blend_weights]
for base_seed in [123, 456, 789, 2024, 2025][:N_SEEDS-1]:
    tp = np.zeros((len(X_test_encoded), n_models))
    for fold, (train_idx, val_idx) in enumerate(cv.split(X_encoded, y_full)):
        X_tr, X_val = X_encoded.iloc[train_idx], X_encoded.iloc[val_idx]
        y_tr = y_full.iloc[train_idx]
        sw_tr = sample_weights[train_idx]
        models = get_models(base_seed, fold)
        for i, m in enumerate(models):
            m.fit(X_tr, y_tr, sample_weight=sw_tr)
            tp[:, i] += m.predict_proba(X_test_encoded)[:, 1] / N_FOLDS
    all_test_preds.append(tp @ blend_weights)

final_preds = np.mean(all_test_preds, axis=0)
submission = pd.DataFrame({'id': test_ids, 'Churn': final_preds})
submission.to_csv('submission.csv', index=False)
submission.head()

,id,Churn
0,594194,0.147194
1,594195,0.001293
2,594196,0.193608
3,594197,0.008067
4,594198,0.663897
